# POS-conditioned geometry 

In [1]:
from __future__ import annotations

import ast
import random
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer

In [2]:
try:
    from IsoScore import IsoScore 
except Exception as e:
    raise ImportError(
        "IsoScore is required (real metric, no fallback). Install with: pip install IsoScore"
    ) from e

try:
    from dadapy import Data  
except Exception as e:
    raise ImportError(
        "dadapy is required (real TwoNN/GRIDE). Install with: pip install dadapy"
    ) from e

try:
    from skdim.id import MOM, TLE, CorrInt, FisherS, lPCA, MLE, MADA, ESS, KNN
except Exception as e:
    raise ImportError(
        "scikit-dimension is required (real ID estimators). Install with: pip install scikit-dimension"
    ) from e

In [3]:
RAND_SEED = 42
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)
torch.manual_seed(RAND_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [18]:
CSV_PATH = "data/en_ewt-ud-train_sentences.csv"
TOKENS_COL = "tokens"
POS_COL    = "pos"    

INDEX_COL = "index"    

MODELS = [
    # (model_name, tag, word_rep_mode)
    ("bert-base-uncased", "bert", "first"),
    ("gpt2",              "gpt2", "last"),
]

MAX_LENGTH = 256
BATCH_SIZE_SENT = 8

MAX_TOKENS_PER_POS = 100       #in paper it was not capped
DROP_FIRST_WORD_FOR_DECODER = True  

N_BOOTSTRAP_FAST  = 50        # for spectral / fast metrics
N_BOOTSTRAP_HEAVY = 200       # for kNN / heavy metrics
FAST_M_MAX  = 4000            # sample size per bootstrap replicate (fast)
HEAVY_M_MAX = 1500            # keep smaller for kNN-based estimators

DADAPY_GRID_RANGE_MAX = 64



METRICS_FAST  = ["iso", "spect", "rand", "sf", "vmf_kappa", "erank", "pr", "stable_rank", "pca", "pca99"]
METRICS_HEAVY = ["twonn", "gride", "mom", "tle", "corrint", "fishers", "mle", "mada", "ess"]

In [19]:
def _to_list(x):
    """Robustly parse list-like columns (already-list or python-literal string)."""
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else [v]
        except Exception:
            # fallback: whitespace split
            return s.split()
    return list(x)


In [20]:
def load_sentence_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "sentence_id" not in df.columns:
        raise ValueError("CSV must contain a 'sentence_id' column.")
    if TOKENS_COL not in df.columns:
        raise ValueError(f"CSV must contain '{TOKENS_COL}' column with list[str].")
    if POS_COL not in df.columns:
        raise ValueError(f"CSV must contain '{POS_COL}' column with list[str] UPOS tags.")

    df = df.copy()
    df["sentence_id"] = df["sentence_id"].astype(str)
    df[TOKENS_COL] = df[TOKENS_COL].apply(_to_list)
    df[POS_COL] = df[POS_COL].apply(_to_list)

    if INDEX_COL in df.columns:
        df[INDEX_COL] = df[INDEX_COL].apply(_to_list)

    return df


In [21]:
def explode_tokens(df_sent: pd.DataFrame) -> pd.DataFrame:
    """Sentence-level df -> token-level df with (sentence_id, word_id, token, pos, index?)."""
    rows = []
    has_index = INDEX_COL in df_sent.columns

    for _, row in df_sent.iterrows():
        sid = str(row["sentence_id"])
        toks = row[TOKENS_COL]
        poss = row[POS_COL]
        idxs = row[INDEX_COL] if has_index else None

        if not isinstance(toks, list) or not isinstance(poss, list):
            continue

        n = min(len(toks), len(poss))
        if has_index and isinstance(idxs, list):
            n = min(n, len(idxs))

        for wid in range(n):
            rows.append({
                "sentence_id": sid,
                "word_id": wid,
                "token": toks[wid],
                "pos": poss[wid],
                "index": (idxs[wid] if has_index and isinstance(idxs, list) else None),
            })

    return pd.DataFrame(rows)


In [22]:
df_sent = load_sentence_df(CSV_PATH)
tok_df  = explode_tokens(df_sent)

print("Sentences:", len(df_sent))
print("Tokens (exploded):", len(tok_df))
print("POS classes:", tok_df["pos"].nunique())

tok_df["pos"].value_counts().head(12)

Sentences: 10067
Tokens (exploded): 194916
POS classes: 17


pos
NOUN     33607
PUNCT    22123
VERB     22095
PRON     18255
ADP      17557
DET      16123
ADJ      12648
AUX      11526
PROPN    11203
ADV       9913
CCONJ     6627
PART      4241
Name: count, dtype: int64

In [23]:
def build_tokenizer_and_model(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tok.pad_token is None:
        if tok.eos_token is None:
            raise ValueError(f"Tokenizer for {model_name} has no pad_token and no eos_token; set a pad token manually.")
        tok.pad_token = tok.eos_token
    if not getattr(tok, "is_fast", False):
        raise ValueError(
            f"Tokenizer for {model_name} is not a fast tokenizer; word alignment requires a fast tokenizer."
        )
    cfg = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
    model = AutoModel.from_pretrained(model_name, config=cfg)
    model.eval().to(DEVICE)
    return tok, model


def _is_decoder_like(model_name: str) -> bool:
    mn = model_name.lower()
    return any(k in mn for k in ["gpt", "opt", "llama", "mistral", "phi", "bloom"])

In [24]:
@torch.no_grad()
def embed_subset_all_layers(
    df_sent: pd.DataFrame,
    subset_df: pd.DataFrame,
    tokenizer,
    model,
    word_rep_mode: str,
    batch_size: int = 8,
    max_length: int = 256,
) -> Tuple[np.ndarray, np.ndarray]:

    subset_df = subset_df.copy()
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    df_sent = df_sent.copy()
    df_sent["sentence_id"] = df_sent["sentence_id"].astype(str)

    by_sid: Dict[str, List[Tuple[int, int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id", "word_id"]].itertuples(index=False, name=None)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    sids = list(by_sid.keys())
    if len(sids) == 0:
        raise ValueError("Empty subset_df")

    df_sel = (
        df_sent[df_sent["sentence_id"].isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    reps = None
    filled = np.zeros(len(subset_df), dtype=bool)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )

    for start in tqdm(range(0, len(sids), batch_size), desc="Embedding"):
        batch_ids = sids[start:start + batch_size]
        toks_batch = df_sel.loc[batch_ids, TOKENS_COL].tolist()

        enc = tokenizer(toks_batch, **enc_kwargs)

        word_ids_list = [enc.word_ids(batch_index=bi) for bi in range(len(batch_ids))]

        enc = enc.to(DEVICE)
        out = model(**enc)
        hs = out.hidden_states  

        if reps is None:
            L = len(hs)
            D = hs[0].shape[-1]
            reps = np.zeros((L, len(subset_df), D), dtype=np.float32)

        for bi, sid in enumerate(batch_ids):
            word_ids = word_ids_list[bi]
            pos_map: Dict[int, List[int]] = {}
            for ti, wid in enumerate(word_ids):
                if wid is not None:
                    pos_map.setdefault(int(wid), []).append(int(ti))

            for gidx, wid in by_sid[str(sid)]:
                if wid not in pos_map:
                    continue
                positions = pos_map[wid]

                if word_rep_mode == "first":
                    tok_pos = positions[0]
                    for l in range(len(hs)):
                        reps[l, gidx, :] = hs[l][bi, tok_pos, :].detach().float().cpu().numpy()
                elif word_rep_mode == "last":
                    tok_pos = positions[-1]
                    for l in range(len(hs)):
                        reps[l, gidx, :] = hs[l][bi, tok_pos, :].detach().float().cpu().numpy()
                elif word_rep_mode == "mean":
                    idx_t = torch.tensor(positions, device=hs[0].device)
                    for l in range(len(hs)):
                        reps[l, gidx, :] = hs[l][bi, idx_t, :].mean(dim=0).detach().float().cpu().numpy()
                else:
                    raise ValueError("word_rep_mode must be one of {'first','last','mean'}")

                filled[gidx] = True

        del out, hs
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    assert reps is not None
    return reps, filled

In [25]:
EPS = 1e-9

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    _, S, _ = np.linalg.svd(Xc, full_matrices=False)
    lam = (S ** 2).astype(np.float64)
    lam.sort()
    return lam[::-1]

def _jitter_unique(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype, copy=False)
    except Exception:
        pass
    return X

In [26]:
def _iso_once(X: np.ndarray) -> float:
    return float(IsoScore.IsoScore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2:
        return float("nan")
    rng = np.random.default_rng()
    K_eff = min(K, (n * (n - 1)) // 2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A * B, axis=1)
    den = (np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num / den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2:
        return float('nan')
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9:
        return 0.0
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    c = np.cumsum(lam)
    thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _pca95_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.95)

def _pca99_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.99)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    s1 = lam.sum()
    s2 = (lam ** 2).sum()
    return float((s1 ** 2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    return float(lam.sum() / (lam.max() + EPS))


In [27]:
def _dadapy_twonn_once(X: np.ndarray) -> float:
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    d = Data(coordinates=_jitter_unique(X))
    d.compute_distances(maxk=DADAPY_GRID_RANGE_MAX)
    ids, _, _ = d.return_id_scaling_gride(range_max=DADAPY_GRID_RANGE_MAX)
    return float(ids[-1])

In [28]:
def _skdim_factory(name: str):
    mapping = {
        'mom': MOM,
        'tle': TLE,
        'corrint': CorrInt,
        'fishers': FisherS,
        'mle': MLE,
        'mada': MADA,
        'ess': ESS,
        'knn': KNN,
        'pca': lPCA,
        'pca99': lPCA,
    }
    cls = mapping.get(name)
    if cls is None:
        return None

    def _builder():
        if name == 'pca':
            return cls(ver='FO')
        if name == 'pca99':
            return cls(ver='ratio', alphaRatio=0.99)
        return cls()

    return _builder

def _skdim_once_builder(name: str) -> Callable[[np.ndarray], float]:
    build = _skdim_factory(name)
    if build is None:
        raise ValueError(f'Unknown skdim estimator: {name}')

    def _once(X: np.ndarray) -> float:
        est = build()
        est.fit(_jitter_unique(X))
        return float(getattr(est, 'dimension_', float('nan')))
    return _once



In [29]:
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    'iso': _iso_once,
    'spect': _spect_once,
    'rand': _rand_once,
    'sf': _sf_once,
    'vmf_kappa': _vmf_kappa_once,
    'erank': _erank_once,
    'pr': _pr_once,
    'stable_rank': _stable_rank_once,
}

HEAVY_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    'twonn': _dadapy_twonn_once,
    'gride': _dadapy_gride_once,
    'mom': _skdim_once_builder('mom'),
    'tle': _skdim_once_builder('tle'),
    'corrint': _skdim_once_builder('corrint'),
    'fishers': _skdim_once_builder('fishers'),
    'mle': _skdim_once_builder('mle'),
    'mada': _skdim_once_builder('mada'),
    'ess': _skdim_once_builder('ess'),
    'pca': _skdim_once_builder('lpca'),
    'pca99': _skdim_once_builder('lpca'),
}

LABELS = {
    # Isotropy
    'iso': 'IsoScore',
    'spect': 'Spectral Ratio',
    'rand': 'RandCos |μ|',
    'sf': 'Spectral Flatness',
    'vmf_kappa': 'vMF κ',
    # Linear ID
    'erank': 'Effective Rank',
    'pr': 'Participation Ratio',
    'stable_rank': 'Stable Rank',
    # Noninear ID
    'twonn': 'TwoNN',
    'gride': 'GRIDE',
    'mom': 'MOM',
    'tle': 'TLE',
    'corrint': 'CorrInt',
    'fishers': 'FisherS',
    'mle': 'MLE',
    'mada': 'MADA',
    'ess': 'ESS',
    'pca': 'PCA FO',
    'pca99': 'PCA@0.99',
}

In [30]:
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap per-layer metric with replacement sampling."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    M = min(M, N)
    A = np.full((n_reps, L), np.nan, np.float32)
    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan

    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

In [31]:
def compute_pos_curves_for_model(
    model_name: str,
    model_tag: str,
    rep_mode: str,
    classes: List[str],
) -> Dict[str, Dict[str, Dict[str, np.ndarray]]]:
    """
    Returns:
      results[metric][pos_class] = {'mu': (L,), 'lo': (L,), 'hi': (L,)}
    """
    tokenizer, model = build_tokenizer_and_model(model_name)

    results: Dict[str, Dict[str, Dict[str, np.ndarray]]] = {m: {} for m in (METRICS_FAST + METRICS_HEAVY)}

    tok_df_use = tok_df
    if DROP_FIRST_WORD_FOR_DECODER and _is_decoder_like(model_name):
        tok_df_use = tok_df_use[tok_df_use['word_id'] != 0].reset_index(drop=True)

    for pos_class in tqdm(classes, desc=f"{model_tag}: POS classes"):
        sub = tok_df_use[tok_df_use['pos'] == pos_class]
        if len(sub) == 0:
            continue
        if len(sub) > MAX_TOKENS_PER_POS:
            sub = sub.sample(n=MAX_TOKENS_PER_POS, random_state=RAND_SEED).reset_index(drop=True)

        reps, filled = embed_subset_all_layers(
            df_sent=df_sent,
            subset_df=sub[['sentence_id','word_id']],
            tokenizer=tokenizer,
            model=model,
            word_rep_mode=rep_mode,
            batch_size=BATCH_SIZE_SENT,
            max_length=MAX_LENGTH,
        )

        good = np.where(filled)[0]
        reps = reps[:, good, :]

        for metric in METRICS_FAST:
            mu, lo, hi = _bs_layer_loop(
                reps, M=FAST_M_MAX, n_reps=N_BOOTSTRAP_FAST, compute_once=FAST_ONCE[metric]
            )
            results[metric][pos_class] = {'mu': mu, 'lo': lo, 'hi': hi}

        for metric in METRICS_HEAVY:
            mu, lo, hi = _bs_layer_loop(
                reps, M=HEAVY_M_MAX, n_reps=N_BOOTSTRAP_HEAVY, compute_once=HEAVY_ONCE[metric]
            )
            results[metric][pos_class] = {'mu': mu, 'lo': lo, 'hi': hi}

        del reps
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return results


pos_counts = tok_df['pos'].value_counts()
POS_CLASSES = pos_counts.index.tolist()

print('POS classes:', POS_CLASSES)

all_results = {}
for model_name, tag, rep_mode in MODELS:
    all_results[tag] = compute_pos_curves_for_model(model_name, tag, rep_mode, POS_CLASSES)



POS classes: ['NOUN', 'PUNCT', 'VERB', 'PRON', 'ADP', 'DET', 'ADJ', 'AUX', 'PROPN', 'ADV', 'CCONJ', 'PART', 'SCONJ', 'NUM', 'SYM', 'INTJ', 'X']


bert: POS classes:   0%|          | 0/17 [00:00<?, ?it/s]

Embedding:   0%|          | 0/13 [00:00<?, ?it/s]

KeyError: 'pca'

In [ ]:
PLOT_METRICS = ['iso', 'pca99', 'gride']

n_rows = len(PLOT_METRICS)
n_cols = len(MODELS)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.6*n_cols, 3.3*n_rows), sharex=True)
if n_rows == 1:
    axes = np.array([axes])
if n_cols == 1:
    axes = axes[:, None]

# Stable palette over classes
cmap = plt.get_cmap('tab20')
palette = {c: cmap(i % 20) for i, c in enumerate(POS_CLASSES)}

layers = None
for col, (model_name, tag, rep_mode) in enumerate(MODELS):
    for row, metric in enumerate(PLOT_METRICS):
        ax = axes[row, col]
        metric_res = all_results[tag].get(metric, {})
        if not metric_res:
            ax.set_axis_off()
            continue

        if layers is None:
            any_cls = next(iter(metric_res.keys()))
            layers = np.arange(metric_res[any_cls]['mu'].shape[0])

        for cls in POS_CLASSES:
            if cls not in metric_res:
                continue
            mu = metric_res[cls]['mu']
            lo = metric_res[cls]['lo']
            hi = metric_res[cls]['hi']
            ax.plot(layers, mu, lw=1.4, color=palette[cls], label=cls)
            ax.fill_between(layers, lo, hi, alpha=0.12, color=palette[cls])

        ax.set_title(f"{tag} — {LABELS.get(metric, metric)} ({rep_mode})")
        ax.set_xlabel('Layer')
        ax.set_ylabel(LABELS.get(metric, metric))

# Legend once (right side)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center left', bbox_to_anchor=(0.99, 0.5), frameon=False, fontsize='small')
plt.tight_layout(rect=[0, 0, 0.92, 1])

if SAVE_FIG:
    plt.savefig(FIG_PATH, dpi=220)

plt.show()